1. Tự động đọc dữ liệu geojson từ nguồn online --> tên tỉnh --> huyện --> xã (vị trí địa lý Geometry) --> lưu vào file vietnam_communes_epsg4326.geojson
Kiểm tra & Chuẩn hóa Hệ quy chiếu (CRS) về EPSG:4326

In [ ]:
import os
import zipfile
import requests
import geopandas as gpd
from shapely.validation import make_valid

# -------------------------------------------------------------
# 1. Cấu hình đường dẫn & URL GADM 4.1 cho Việt Nam
# -------------------------------------------------------------
DATA_DIR = "./data_spatial"
ZIP_PATH = os.path.join(DATA_DIR, "gadm41_VNM_shp.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "gadm41_VNM")
OUTPUT_CLEAN_PATH = os.path.join(DATA_DIR, "vietnam_communes_epsg4326.geojson")

os.makedirs(DATA_DIR, exist_ok=True)

# URL chính thức của GADM 4.1 cho gói Shapefile Việt Nam
GADM_VNM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_VNM_shp.zip"

# -------------------------------------------------------------
# 2. Tải và giải nén dữ liệu
# -------------------------------------------------------------
if not os.path.exists(ZIP_PATH):
    print(f"[*] Đang tải dữ liệu GADM từ: {GADM_VNM_URL}...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(GADM_VNM_URL, headers=headers, stream=True)
    response.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print("[+] Tải thành công.")
else:
    print("[*] File nén đã tồn tại, bỏ qua bước tải.")

if not os.path.exists(EXTRACT_DIR):
    print("[*] Đang giải nén file...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("[+] Giải nén hoàn tất.")

# -------------------------------------------------------------
# 3. Đọc dữ liệu Level 3 (Cấp Xã/Phường/Thị trấn)
# -------------------------------------------------------------
shp_level3_path = os.path.join(EXTRACT_DIR, "gadm41_VNM_3.shp")
print(f"[*] Đang nạp Shapefile cấp xã: {shp_level3_path}")
gdf_communes = gpd.read_file(shp_level3_path)

# -------------------------------------------------------------
# 4. Kiểm tra & Chuẩn hóa Hệ quy chiếu (CRS) về EPSG:4326
# -------------------------------------------------------------
target_crs = "EPSG:4326"

print(f"[*] CRS ban đầu: {gdf_communes.crs}")
if gdf_communes.crs is None:
    print("[!] Dữ liệu không có thông tin CRS, thiết lập mặc định EPSG:4326.")
    gdf_communes = gdf_communes.set_crs(target_crs)
elif gdf_communes.crs.to_string() != target_crs:
    print(f"[*] Chuyển đổi CRS từ {gdf_communes.crs} -> {target_crs}...")
    gdf_communes = gdf_communes.to_crs(target_crs)
else:
    print("[+] Dữ liệu đã ở định dạng chuẩn WGS 84 (EPSG:4326).")

# -------------------------------------------------------------
# 5. Sửa lỗi đa giác hình học (Geometry Cleaning & Validation)
# -------------------------------------------------------------
# Dữ liệu viễn thám dễ bị crash nếu đa giác xã bị tự cắt (self-intersection)
invalid_count = (~gdf_communes.is_valid).sum()
if invalid_count > 0:
    print(f"[!] Phát hiện {invalid_count} đa giác xã bị lỗi hình học. Đang tự động sửa...")
    gdf_communes["geometry"] = gdf_communes["geometry"].apply(make_valid)

# -------------------------------------------------------------
# 6. Kiểm tra cấu trúc thuộc tính và lưu file sạch
# -------------------------------------------------------------
cols_to_keep = ["GID_1", "NAME_1", "GID_2", "NAME_2", "GID_3", "NAME_3", "TYPE_3", "geometry"]
gdf_cleaned = gdf_communes[cols_to_keep].copy()

# Xuất ra GeoJSON chuẩn để dễ nạp lên GEE Asset hoặc chạy trên PyTorch
gdf_cleaned.to_file(OUTPUT_CLEAN_PATH, driver="GeoJSON")

print("\n--- KẾT QUẢ KIỂM TRA DỮ LIỆU ---")
print(f"- Tổng số đơn vị cấp xã: {len(gdf_cleaned):,}")
print(f"- Hệ tọa độ cuối cùng: {gdf_cleaned.crs}")
print(f"- Các cột định danh: {list(gdf_cleaned.columns[:-1])}")
print(f"- File lưu tại: {OUTPUT_CLEAN_PATH}")

In [ ]:
# Check
print(gdf_cleaned.shape)
print(gdf_cleaned.head(5))
print(gdf_cleaned[gdf_cleaned["GID_3"]== "VNM.27.1.1_1"])

2. KẾT NỐI GOOLE EARTH ENGINE

In [ ]:
import ee

# 1. Kích hoạt xác thực (nếu đã xác thực rồi thì cell sẽ tự bỏ qua)
try:
    ee.Initialize(project="angelic-digit-434804-r9")
    print("[+] GEE đã được khởi tạo thành công!")
except Exception as e:
    print("[*] Yêu cầu xác thực tài khoản Google...")
    ee.Authenticate()
    ee.Initialize(project="angelic-digit-434804-r9")
    print("[+] Xác thực và khởi tạo GEE hoàn tất!")

In [ ]:
import calendar
from datetime import datetime
import os
import re
import unicodedata
import ee

PROJECT_ID = "angelic-digit-434804-r9"
ASSET_ID = f"projects/{PROJECT_ID}/assets/vietnam_communes_l3"

# 1. Khởi tạo GEE
try:
  ee.Initialize(project=PROJECT_ID)
  print("[+] GEE đã được khởi tạo thành công!")
except Exception:
  ee.Authenticate()
  ee.Initialize(project=PROJECT_ID)

communes_fc = ee.FeatureCollection(ASSET_ID)


def normalize_str(s):
  if not s:
    return ""
  s = unicodedata.normalize("NFD", str(s))
  s = re.sub(r"[\u0300-\u036f]", "", s)
  s = s.replace("đ", "d").replace("Đ", "d")
  s = re.sub(r"[^a-zA-Z0-9]+", "_", s)
  return s.strip("_").lower()


def natural_sort_key(gid_str):
  return [
      int(text) if text.isdigit() else text.lower()
      for text in re.split(r"(\d+)", str(gid_str))
  ]


# 2. Xử lý ảnh VIIRS Nighttime Lights Monthly
# Bộ dữ liệu: NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG (Đã lọc ánh sáng tán xạ mặt trăng & đám mây)
def get_viirs_monthly_composite(year, month, commune_geom):
  _, last_day = calendar.monthrange(year, month)
  s_date = f"{year}-{month:02d}-01"
  e_date = f"{year}-{month:02d}-{last_day:02d}"

  col = (
      ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
      .filterBounds(commune_geom)
      .filterDate(s_date, e_date)
  )

  # Nếu bộ lọc VCMSLCFG rỗng (thường ở các tháng biên), dự phòng qua VCMCFG
  if col.size().getInfo() == 0:
    col = (
        ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")
        .filterBounds(commune_geom)
        .filterDate(s_date, e_date)
    )

  if col.size().getInfo() > 0:
    # avg_rad: Bức xạ ánh sáng đêm trung bình
    # cf_cvg: Số lượt quan sát sạch mây trong tháng
    return (
        col.select(["avg_rad", "cf_cvg"])
        .mean()
        .clip(commune_geom)
        .set("system:time_start", s_date)
    )
  return None


# 3. Trích xuất danh mục Tỉnh (Ưu tiên gdf_cleaned trong RAM)
prov_map = {}
if "gdf_cleaned" in globals() and "NAME_1" in gdf_cleaned.columns:
  print("[*] Đang đọc danh mục tỉnh từ 'gdf_cleaned'...")
  unique_provs = (
      gdf_cleaned[["NAME_1", "GID_1"]].drop_duplicates().dropna().values
  )
  for name, gid1 in unique_provs:
    prov_map[str(name)] = str(gid1)
else:
  print("[*] Đang đọc danh mục tỉnh từ Asset GEE...")
  sample_props = communes_fc.first().getInfo()["properties"]
  col_pname = "NAME_1" if "NAME_1" in sample_props else None
  if col_pname:
    data = communes_fc.reduceColumns(
        ee.Reducer.toList(2), [col_pname, "GID_3"]
    ).getInfo()["list"]
    for name, gid3 in data:
      if name and gid3:
        parts = str(gid3).split(".")
        if len(parts) >= 2:
          gid1 = f"{parts[0]}.{parts[1]}_1"
          prov_map[str(name)] = gid1

if not prov_map:
  gid3_list = communes_fc.aggregate_array("GID_3").getInfo()
  distinct_gids = sorted(
      list(
          set(
              [
                  f"{x.split('.')[0]}.{x.split('.')[1]}_1"
                  for x in gid3_list
                  if "." in str(x)
              ]
          )
      )
  )
  for gid in distinct_gids:
    prov_map[gid] = gid

# 4. Hiển thị danh mục Tỉnh
print("=" * 88)
print(f"[+] DANH MỤC {len(prov_map)} TỈNH/THÀNH PHỐ KÈM MÃ PROVINCE:")
items = [f"{name} ({gid})" for name, gid in sorted(prov_map.items())]
for i in range(0, len(items), 2):
  row_items = items[i : i + 2]
  print("  ".join(f"{item:<42}" for item in row_items))
print("=" * 88)

# 5. Người dùng chọn Tỉnh
user_prov = input("\n👉 1. Nhập TÊN hoặc MÃ tỉnh/thành phố: ").strip()
norm_prov = normalize_str(user_prov)

matched_prov_name = None
for official_name, gid in prov_map.items():
  if (
      normalize_str(official_name) == norm_prov
      or normalize_str(gid) == norm_prov
  ):
    matched_prov_name = official_name
    break

if not matched_prov_name:
  print(f"\n[-] LỖI: Không tìm thấy tỉnh nào khớp với '{user_prov}'.")
else:
  target_prov_gid = prov_map[matched_prov_name]
  print(f"[+] Đã chọn tỉnh: '{matched_prov_name}' (Mã: {target_prov_gid})")

  # 6. Tải danh sách xã
  print(f"[*] Đang tải danh sách xã thuộc {matched_prov_name}...")
  prov_prefix = target_prov_gid.replace("_1", "") + "."
  commune_map = {}

  if "gdf_cleaned" in globals() and "NAME_3" in gdf_cleaned.columns:
    df_sub = gdf_cleaned[gdf_cleaned["GID_3"].str.startswith(prov_prefix)]
    for _, row in df_sub.iterrows():
      gid3 = str(row["GID_3"])
      c_type = str(row.get("TYPE_3", "")).strip()
      c_name = str(row["NAME_3"]).strip()
      full_name = (
          f"{c_type} {c_name}"
          if c_type and c_type.lower() != "nan"
          else c_name
      )
      commune_map[gid3] = full_name
  else:
    prov_communes = communes_fc.filter(
        ee.Filter.stringStartsWith("GID_3", prov_prefix)
    )
    first_feat = prov_communes.first().getInfo()
    available_cols = (
        list(first_feat["properties"].keys()) if first_feat else []
    )
    possible_name_cols = [
        "NAME_3",
        "NAME_3_vi",
        "VARNAME_3",
        "ADM3_VI",
        "NAME_2",
    ]
    name_col = next(
        (col for col in possible_name_cols if col in available_cols), None
    )
    type_col = "TYPE_3" if "TYPE_3" in available_cols else None

    features = prov_communes.getInfo()["features"]
    for f in features:
      props = f["properties"]
      gid3 = props.get("GID_3")
      c_name = props.get(name_col) if name_col else None
      c_type = props.get(type_col, "") if type_col else ""
      full_name = (
          f"{c_type} {c_name}".strip() if c_name and c_type else (c_name or gid3)
      )
      commune_map[gid3] = full_name

  # Sắp xếp Natural Sort theo GID_3
  sorted_by_gid = sorted(
      commune_map.items(), key=lambda x: natural_sort_key(x[0])
  )

  print("=" * 88)
  print(
      f"[+] DANH SÁCH {len(commune_map)} XÃ/PHƯỜNG THUỘC"
      f" {matched_prov_name.upper()} (ORDER THEO MÃ GID_3):"
  )
  c_items = [f"{c_name} ({gid})" for gid, c_name in sorted_by_gid]
  for i in range(0, len(c_items), 2):
    row_items = c_items[i : i + 2]
    print("  ".join(f"{item:<42}" for item in row_items))
  print("=" * 88)

  # 7. Người dùng chọn Xã
  user_commune = input(
      "\n👉 2. Nhập TÊN hoặc MÃ XÃ (GID_3) cần trích xuất: "
  ).strip()
  norm_commune = normalize_str(user_commune)

  matched_gid3 = None
  matched_cname = None
  for gid, c_name in commune_map.items():
    if (
        normalize_str(c_name) == norm_commune
        or normalize_str(gid) == norm_commune
    ):
      matched_gid3 = gid
      matched_cname = c_name
      break

  if not matched_gid3:
    print(f"\n[-] LỖI: Không tìm thấy xã nào khớp với '{user_commune}'.")
  else:
    print(f"[+] XÁC NHẬN: {matched_cname} (Mã: {matched_gid3})")

    commune_geom = communes_fc.filter(
        ee.Filter.eq("GID_3", matched_gid3)
    ).geometry()

    clean_pname = normalize_str(matched_prov_name)
    clean_cname = normalize_str(matched_cname)
    safe_gid3 = matched_gid3.replace(".", "_")

    # VIIRS DNB có dữ liệu liên tục từ tháng 04/2012 đến nay
    start_year = 2016
    start_month = 1
    current_year = datetime.now().year
    current_month = datetime.now().month

    time_start_str = f"{start_year}{start_month:02d}"
    time_end_str = f"{current_year}{current_month:02d}"

    # QUY TẮC ĐẶT TÊN THƯ MỤC
    drive_folder = f"VIIRS_Night_{clean_pname}_{clean_cname}_{safe_gid3}_{time_start_str}-{time_end_str}"

    print("\n" + "=" * 80)
    print(
        f"[*] THIẾT LẬP XUẤT ẢNH ÁNH SÁNG ĐÊM ({time_start_str} ->"
        f" {time_end_str}):"
    )
    print(f"    - Thư mục Google Drive: {drive_folder}")
    print(
        f"    - Quy chuẩn tên file:  "
        f" VIIRS_Night_{clean_pname}_{clean_cname}_{safe_gid3}_[YYYYMM]"
    )
    print(
        f"    - Kênh dữ liệu:         Băng avg_rad (Bức xạ đêm) & cf_cvg (Số"
        f" lượt sạch mây)"
    )
    print(f"    - Độ phân giải ảnh:     500m/pixel (chuẩn vệ tinh VIIRS)")
    print("=" * 80)

    tasks_submitted = 0

    for yr in range(start_year, current_year + 1):
      end_m = current_month if yr == current_year else 12
      for m in range(1, end_m + 1):
        time_point_str = f"{yr}{m:02d}"

        night_img = get_viirs_monthly_composite(yr, m, commune_geom)

        if night_img is not None:
          # QUY TẮC ĐẶT TÊN FILE
          file_pfx = f"VIIRS_Night_{clean_pname}_{clean_cname}_{safe_gid3}_{time_point_str}"
          task_desc = f"VIIRS_{safe_gid3}_{time_point_str}"

          task = ee.batch.Export.image.toDrive(
              image=night_img,
              description=task_desc,
              folder=drive_folder,
              fileNamePrefix=file_pfx,
              region=commune_geom,
              scale=500,  # Độ phân giải gốc của VIIRS DNB
              crs="EPSG:4326",
              maxPixels=1e9,
          )
          task.start()
          tasks_submitted += 1
          print(f"[+] Đã gửi Task: Tháng {m:02d}/{yr} -> File: {file_pfx}")
        else:
          print(
              f"[-] Bỏ qua tháng {m:02d}/{yr}: Không có dữ liệu ánh sáng đêm"
              " VIIRS."
          )

    print("\n" + "=" * 80)
    print(
        f"[+] HOÀN TẤT! Đã gửi tổng cộng {tasks_submitted} tác vụ về thư"
        f" mục:\n    👉 Drive/{drive_folder}"
    )
    print("=" * 80)

In [ ]:
import calendar
from datetime import datetime
import os
import re
import unicodedata
import ee
import pandas as pd

PROJECT_ID = "angelic-digit-434804-r9"
ASSET_ID = f"projects/{PROJECT_ID}/assets/vietnam_communes_l3"

# 1. Khởi tạo GEE
try:
  ee.Initialize(project=PROJECT_ID)
  print("[+] GEE đã được khởi tạo thành công!")
except Exception:
  ee.Authenticate()
  ee.Initialize(project=PROJECT_ID)

communes_fc = ee.FeatureCollection(ASSET_ID)


def normalize_str(s):
  if not s:
    return ""
  s = unicodedata.normalize("NFD", str(s))
  s = re.sub(r"[\u0300-\u036f]", "", s)
  s = s.replace("đ", "d").replace("Đ", "d")
  s = re.sub(r"[^a-zA-Z0-9]+", "_", s)
  return s.strip("_").lower()


def natural_sort_key(gid_str):
  return [
      int(text) if text.isdigit() else text.lower()
      for text in re.split(r"(\d+)", str(gid_str))
  ]


# 2. Trích xuất danh mục Tỉnh (NAME_1 <-> GID_1)
prov_map = {}
if "gdf_cleaned" in globals() and "NAME_1" in gdf_cleaned.columns:
  print("[*] Đang nạp danh mục tỉnh từ 'gdf_cleaned'...")
  unique_provs = (
      gdf_cleaned[["NAME_1", "GID_1"]].drop_duplicates().dropna().values
  )
  for name, gid1 in unique_provs:
    prov_map[str(name)] = str(gid1)
else:
  print("[*] Đang đọc danh mục tỉnh từ Asset GEE...")
  sample_props = communes_fc.first().getInfo()["properties"]
  col_pname = "NAME_1" if "NAME_1" in sample_props else None
  if col_pname:
    data = communes_fc.reduceColumns(
        ee.Reducer.toList(2), [col_pname, "GID_3"]
    ).getInfo()["list"]
    for name, gid3 in data:
      if name and gid3:
        parts = str(gid3).split(".")
        if len(parts) >= 2:
          gid1 = f"{parts[0]}.{parts[1]}_1"
          prov_map[str(name)] = gid1

if not prov_map:
  gid3_list = communes_fc.aggregate_array("GID_3").getInfo()
  distinct_gids = sorted(
      list(
          set(
              [
                  f"{x.split('.')[0]}.{x.split('.')[1]}_1"
                  for x in gid3_list
                  if "." in str(x)
              ]
          )
      )
  )
  for gid in distinct_gids:
    prov_map[gid] = gid

# 3. Hiển thị danh mục Tỉnh
print("=" * 88)
print(f"[+] DANH MỤC {len(prov_map)} TỈNH/THÀNH PHỐ KÈM MÃ PROVINCE:")
items = [f"{name} ({gid})" for name, gid in sorted(prov_map.items())]
for i in range(0, len(items), 2):
  row_items = items[i : i + 2]
  print("  ".join(f"{item:<42}" for item in row_items))
print("=" * 88)

# 4. Người dùng chọn Tỉnh
user_prov = input("\n👉 1. Nhập TÊN hoặc MÃ tỉnh/thành phố: ").strip()
norm_prov = normalize_str(user_prov)

matched_prov_name = None
for official_name, gid in prov_map.items():
  if (
      normalize_str(official_name) == norm_prov
      or normalize_str(gid) == norm_prov
  ):
    matched_prov_name = official_name
    break

if not matched_prov_name:
  print(f"\n[-] LỖI: Không tìm thấy tỉnh nào khớp với '{user_prov}'.")
else:
  target_prov_gid = prov_map[matched_prov_name]
  print(f"[+] Đã chọn tỉnh: '{matched_prov_name}' (Mã: {target_prov_gid})")

  # 5. Tải danh sách xã
  print(f"[*] Đang tải danh sách xã thuộc {matched_prov_name}...")
  prov_prefix = target_prov_gid.replace("_1", "") + "."
  commune_map = {}

  if "gdf_cleaned" in globals() and "NAME_3" in gdf_cleaned.columns:
    df_sub = gdf_cleaned[gdf_cleaned["GID_3"].str.startswith(prov_prefix)]
    for _, row in df_sub.iterrows():
      gid3 = str(row["GID_3"])
      c_type = str(row.get("TYPE_3", "")).strip()
      c_name = str(row["NAME_3"]).strip()
      full_name = (
          f"{c_type} {c_name}"
          if c_type and c_type.lower() != "nan"
          else c_name
      )
      commune_map[gid3] = full_name
  else:
    prov_communes = communes_fc.filter(
        ee.Filter.stringStartsWith("GID_3", prov_prefix)
    )
    first_feat = prov_communes.first().getInfo()
    available_cols = (
        list(first_feat["properties"].keys()) if first_feat else []
    )
    possible_name_cols = [
        "NAME_3",
        "NAME_3_vi",
        "VARNAME_3",
        "ADM3_VI",
        "NAME_2",
    ]
    name_col = next(
        (col for col in possible_name_cols if col in available_cols), None
    )
    type_col = "TYPE_3" if "TYPE_3" in available_cols else None

    features = prov_communes.getInfo()["features"]
    for f in features:
      props = f["properties"]
      gid3 = props.get("GID_3")
      c_name = props.get(name_col) if name_col else None
      c_type = props.get(type_col, "") if type_col else ""
      full_name = (
          f"{c_type} {c_name}".strip() if c_name and c_type else (c_name or gid3)
      )
      commune_map[gid3] = full_name

  # Sắp xếp Natural Sort theo GID_3
  sorted_by_gid = sorted(
      commune_map.items(), key=lambda x: natural_sort_key(x[0])
  )

  print("=" * 88)
  print(
      f"[+] DANH SÁCH {len(commune_map)} XÃ/PHƯỜNG THUỘC"
      f" {matched_prov_name.upper()} (ORDER THEO MÃ GID_3):"
  )
  c_items = [f"{c_name} ({gid})" for gid, c_name in sorted_by_gid]
  for i in range(0, len(c_items), 2):
    row_items = c_items[i : i + 2]
    print("  ".join(f"{item:<42}" for item in row_items))
  print("=" * 88)

  # 6. Người dùng chọn Xã
  user_commune = input(
      "\n👉 2. Nhập TÊN hoặc MÃ XÃ (GID_3) cần trích xuất: "
  ).strip()
  norm_commune = normalize_str(user_commune)

  matched_gid3 = None
  matched_cname = None
  for gid, c_name in commune_map.items():
    if (
        normalize_str(c_name) == norm_commune
        or normalize_str(gid) == norm_commune
    ):
      matched_gid3 = gid
      matched_cname = c_name
      break

  if not matched_gid3:
    print(f"\n[-] LỖI: Không tìm thấy xã nào khớp với '{user_commune}'.")
  else:
    print(f"[+] XÁC NHẬN: {matched_cname} (Mã: {matched_gid3})")

    target_feat = communes_fc.filter(ee.Filter.eq("GID_3", matched_gid3))
    commune_geom = target_feat.geometry()

    clean_pname = normalize_str(matched_prov_name)
    clean_cname = normalize_str(matched_cname)
    safe_gid3 = matched_gid3.replace(".", "_")

    start_year = 2016
    start_month = 1
    current_year = datetime.now().year
    current_month = datetime.now().month

    time_start_str = f"{start_year}{start_month:02d}"
    time_end_str = f"{current_year}{current_month:02d}"

    print("\n" + "=" * 80)
    print(
        f"[*] BẮT ĐẦU TRÍCH XUẤT CHUỖI CHỈ SỐ ÁNH SÁNG ĐÊM ({time_start_str} ->"
        f" {time_end_str}):"
    )
    print(f"    - Địa bàn:     {matched_cname} ({matched_gid3})")
    print(
        f"    - Chỉ số:      TNL, Mean Radiance, Lit Area, Electrification"
        f" Ratio, CV, Lit-Pop Proxy"
    )
    print("=" * 80)

    # Lấy diện tích thực địa của xã (ha)
    commune_area_ha = (
        commune_geom.area(maxError=1).divide(10000).getInfo()
    )

    records = []

    # Định nghĩa tổ hợp Reducer thống kê đa chỉ số
    reducers = (
        ee.Reducer.sum()
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
        .combine(ee.Reducer.min(), sharedInputs=True)
        .combine(ee.Reducer.max(), sharedInputs=True)
        .combine(ee.Reducer.count(), sharedInputs=True)
    )

    for yr in range(start_year, current_year + 1):
      end_m = current_month if yr == current_year else 12
      for m in range(1, end_m + 1):
        _, last_day = calendar.monthrange(yr, m)
        s_date = f"{yr}-{m:02d}-01"
        e_date = f"{yr}-{m:02d}-{last_day:02d}"

        # 1. Truy vấn ImageCollection VIIRS ban đêm
        col = (
            ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
            .filterBounds(commune_geom)
            .filterDate(s_date, e_date)
        )
        if col.size().getInfo() == 0:
          col = (
              ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")
              .filterBounds(commune_geom)
              .filterDate(s_date, e_date)
          )

        if col.size().getInfo() == 0:
          print(f"  [-] Tháng {m:02d}/{yr}: Không có ảnh vệ tinh.")
          continue

        img = col.mean().clip(commune_geom)

        rad = img.select("avg_rad")  # Độ phát xạ nanoWatts/cm2/sr
        cf_cvg = img.select("cf_cvg")  # Số quan sát sạch mây

        # 2. Xây dựng các lớp dẫn xuất (Derived Layers)
        # Ngưỡng bức xạ thực tế phát hiện có điện / hoạt động đô thị (loại bỏ nhiễu nền < 1.5 nW)
        lit_mask = rad.gte(1.5).rename("is_lit")
        lit_rad = rad.updateMask(lit_mask).rename("lit_rad")

        # 3. Thống kê cấp xã
        # Thống kê toàn cảnh
        stats_all = rad.reduceRegion(
            reducer=reducers,
            geometry=commune_geom,
            scale=500,
            maxPixels=1e9,
            crs="EPSG:4326",
        ).getInfo()

        # Thống kê trên vùng có ánh sáng (đại diện cho phân bố dân cư đô thị tập trung)
        stats_lit = lit_rad.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=commune_geom,
            scale=500,
            maxPixels=1e9,
            crs="EPSG:4326",
        ).getInfo()

        # Đếm số pixel có ánh sáng
        lit_pixel_count = lit_mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=commune_geom,
            scale=500,
            maxPixels=1e9,
            crs="EPSG:4326",
        ).getInfo().get("is_lit", 0)

        # Số lượt sạch mây trung bình
        cloud_free_obs = cf_cvg.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=commune_geom,
            scale=500,
            maxPixels=1e9,
            crs="EPSG:4326",
        ).getInfo().get("cf_cvg", 0)

        # 4. Tính toán các chỉ số kinh tế - xã hội tích hợp
        total_pixels = stats_all.get("avg_rad_count", 0)
        tnl = stats_all.get("avg_rad_sum", 0.0) or 0.0
        mean_rad = stats_all.get("avg_rad_mean", 0.0) or 0.0
        std_rad = stats_all.get("avg_rad_stdDev", 0.0) or 0.0
        min_rad = stats_all.get("avg_rad_min", 0.0) or 0.0
        max_rad = stats_all.get("avg_rad_max", 0.0) or 0.0

        lit_pop_proxy = (
            stats_lit.get("lit_rad", 0.0) or 0.0
        )  # Tổng bức xạ vùng sáng

        # Tỷ lệ tiếp cận điện lưới / thắp sáng
        electrification_ratio = (
            (lit_pixel_count / total_pixels * 100.0) if total_pixels > 0 else 0.0
        )

        # Diện tích chiếu sáng thực tế (pixel 500m ~ 25 ha)
        lit_area_ha = lit_pixel_count * 25.0

        # Chỉ số biến động ánh sáng trong không gian xã (CV = std / mean)
        spatial_cv = (std_rad / mean_rad) if mean_rad > 0 else 0.0

        records.append({
            "GID_1": target_prov_gid,
            "NAME_1": matched_prov_name,
            "GID_3": matched_gid3,
            "NAME_3": matched_cname,
            "YEAR": yr,
            "MONTH": m,
            "TIME": f"{yr}-{m:02d}",
            "COMMUNE_AREA_HA": round(commune_area_ha, 2),
            "TNL": round(tnl, 4),  # Tổng bức xạ toàn xã
            "MEAN_RAD": round(mean_rad, 4),  # Bức xạ trung bình
            "STD_RAD": round(std_rad, 4),
            "MIN_RAD": round(min_rad, 4),
            "MAX_RAD": round(max_rad, 4),
            "SPATIAL_CV": round(
                spatial_cv, 4
            ),  # Độ phân tán/bất bình đẳng ánh sáng
            "LIT_PIXELS": int(lit_pixel_count),
            "LIT_AREA_HA": round(lit_area_ha, 2),  # Diện tích chiếu sáng
            "ELECTRIFICATION_RATIO_PCT": round(
                electrification_ratio, 2
            ),  # Tỷ lệ thắp sáng (%)
            "LIT_POP_PROXY": round(
                lit_pop_proxy, 4
            ),  # Proxy dân cư/kinh tế tập trung
            "CLOUD_FREE_OBS": round(
                cloud_free_obs, 1
            ),  # Số lần quan sát sạch mây
        })

        print(
            f"[+] Tháng {m:02d}/{yr} | TNL: {tnl:8.2f} | Tỷ lệ thắp sáng:"
            f" {electrification_ratio:5.1f}% | Sạch mây: {cloud_free_obs:3.1f}"
            " lượt"
        )

    # 5. Xuất kết quả ra DataFrame và file CSV
    df_ntl = pd.DataFrame(records)

    # Tính biến động chuỗi thời gian (Temporal Rolling Volatility - độ ổn định nguồn điện qua thời gian)
    df_ntl["TNL_MA3"] = df_ntl["TNL"].rolling(window=3, min_periods=1).mean()
    df_ntl["TNL_MOM_GROWTH_PCT"] = df_ntl["TNL"].pct_change() * 100.0

    out_filename = f"VIIRS_Night_{clean_pname}_{clean_cname}_{safe_gid3}_{time_start_str}-{time_end_str}_Economic_Indices.csv"
    out_path = os.path.join("/content", out_filename)

    df_ntl.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print(f"[+] XUẤT FILE THÀNH CÔNG: {out_path}")
    print(
        f"[+] Tổng số tháng thu thập: {len(df_ntl)} tháng | Số trường dữ liệu:"
        f" {df_ntl.shape[1]} cột"
    )
    print("=" * 80)

    display(df_ntl.head(10))